# Clasificación: CatBoost

### Imports y Configuración

In [ ]:
import sys
sys.path.append("/workspace/src")

import pandas as pd
import matplotlib.pyplot as plt

from EstiloGraficos import aplicar_estilo

# Misma paleta y estilo que el dashboard (components/plotly_theme.py)
aplicar_estilo()

# Textos largos completos en las tablas (criterio de selección, hiperparámetros)
pd.set_option("display.max_colwidth", None)

from Modelado import (
    SEED, USAR_GPU, GPU_DISPOSITIVOS, NOMBRES_CLF, ESTRATEGIAS, N_ITER_FASE1,
    cargar_split, separar_x_y, crear_cv, dispositivo_modelo, verificar_smotenc,
    describir_espacio, ejecutar_fase1, formatear_tabla_cv, graficar_estrategias, elegir_mejor, guardar_fase1,
    formatear_hiperparametros, tabla_hiperparametros,
)

CLAVE = "catboost"

In [ ]:
pd.Series({
    "USAR_GPU": USAR_GPU,
    "GPU detectada por librería": ", ".join(f"{lib}: {disp}" for lib, disp in GPU_DISPOSITIVOS.items() if disp) or "ninguna",
    "Dispositivo de entrenamiento": dispositivo_modelo(CLAVE),
}, name=NOMBRES_CLF[CLAVE]).to_frame()

### Datos: Split y Folds

In [ ]:
df = pd.read_csv('/workspace/data/raw/screen_time_mental_health.csv')

df_train, _ = cargar_split(df)   # el test no se usa en esta fase
X_train, y_train = separar_x_y(df_train)
cv = crear_cv(SEED)

pd.DataFrame({
    "Filas": [len(X_train)],
    "Predictores": [X_train.shape[1]],
    "Deprimidos": [int(y_train.sum())],
    "Prevalencia": [round(y_train.mean(), 3)],
}, index=pd.Index(["Train"], name="Conjunto"))

### Estrategias de Manejo del Desbalance

| Estrategia | Qué hace | En CatBoost |
|---|---|---|
| **a) Nativo** | Sin balanceo: el modelo ve la proporción real (16% de positivos). | — |
| **b) Pesos de clase** | Los errores en la clase minoritaria pesan más al entrenar. | `auto_class_weights="Balanced"` |
| **c) SMOTENC** | Genera deprimidos sintéticos a partir de sus vecinos deprimidos más cercanos, hasta igualar las clases. Sin pesos de clase. | `SMOTENC(categorical_features=["sex"], random_state=42)` |

In [ ]:
tabla_clases, tabla_sexo = verificar_smotenc(X_train, y_train, seed=SEED)
display(tabla_clases)
display(tabla_sexo)

### Espacio de Búsqueda

Se exploraron 25 combinaciones aleatorias por estrategia. Se variaron la complejidad del modelo (profundidad de 3 a 7 y entre 200 y 800 iteraciones), la regularización L2 de las hojas (`l2_leaf_reg` de 1 a 10) y la tasa de aprendizaje en escala logarítmica (0.01 a 0.2).

In [ ]:
describir_espacio(CLAVE).style.set_caption(f"Combinaciones por estrategia: {N_ITER_FASE1[CLAVE]}")

### Validación Cruzada por Estrategia

Para cada estrategia se ejecuta un `RandomizedSearchCV` **solo sobre el train**, con los mismos folds y `scoring` PR-AUC (`average_precision`).

In [ ]:
tabla, pipelines, hiperparametros = ejecutar_fase1(CLAVE, X_train, y_train, cv, seed=SEED)

In [ ]:
vista = formatear_tabla_cv(tabla)
vista["Tiempo (s)"] = tabla["Tiempo (s)"].round(0)
vista.rename(index=ESTRATEGIAS)

*Pendiente: interpretar tras ejecutar el notebook.* Comparar la PR-AUC y la ROC-AUC de las estrategias y si sus desviaciones se solapan.

*Pendiente: interpretar tras ejecutar el notebook.* Precision, recall, F1 y accuracy con umbral 0.5 por estrategia; comparar la accuracy con la de predecir siempre “no deprimido” (1 − 0.164 = 0.836).

In [ ]:
# Mejores hiperparámetros por estrategia
tabla_hiperparametros(hiperparametros)

*Pendiente: interpretar tras ejecutar el notebook.* Qué configuración eligió cada estrategia y qué sugiere sobre la complejidad de la señal.

In [ ]:
fig = graficar_estrategias(tabla, NOMBRES_CLF[CLAVE])
plt.show()

*Pendiente: interpretar tras ejecutar el notebook.* Si las barras de PR-AUC se solapan y cómo cambia el punto de operación (precision vs. recall) con umbral 0.5.

### Selección del Mejor Modelo CatBoost

Se elige la estrategia con **mayor PR-AUC media de CV**.

In [ ]:
estrategia_ganadora, explicacion = elegir_mejor(tabla)

pd.Series({
    "Estrategia ganadora": ESTRATEGIAS[estrategia_ganadora],
    "PR-AUC de CV": f"{tabla.loc[estrategia_ganadora, 'PR-AUC media']:.4f} ± {tabla.loc[estrategia_ganadora, 'PR-AUC DE']:.4f}",
    "Hiperparámetros": formatear_hiperparametros(hiperparametros[estrategia_ganadora]),
    "Criterio": explicacion,
}, name=NOMBRES_CLF[CLAVE]).to_frame()

*Pendiente: interpretar tras ejecutar el notebook.* Qué estrategia se eligió, qué tan lejos quedaron las demás frente a la desviación entre folds y qué implica su recall con umbral 0.5 (el umbral se optimiza en la fase 3).

### Guardado

In [ ]:
ganador = guardar_fase1(
    CLAVE, tabla, estrategia_ganadora, pipelines[estrategia_ganadora],
    hiperparametros[estrategia_ganadora], explicacion,
)